# MLVC demo

Run end-to-end video compression with MLVC: download a pretrained model, encode a sample video, reconstruct its frames, and inspect rate–distortion results.

In [ ]:
import os
import sys
import tempfile

sys.path.append(os.path.abspath(os.path.dirname(os.getcwd())))

import json
import torch
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
from pathlib import Path

from src.utils.video_reader import YUVReader
from src.utils.model_factory import create_video_model
from src.utils.stream_helper import get_state_dict, prepare_frame
from src.transforms.functional import yuv_444_to_420, yuv_420_to_444, ycbcr2rgb
from src.metrics.bjontegaard_metric import calculate_bd_rate

## Configuration

Assets like test sequences and model ckpts will be downloaded and stored under `~/mlvc` by default. Edit `MLVC_DIR` to use another location.

In [ ]:
MLVC_DIR = Path("~/mlvc").expanduser()

dataset_dir = MLVC_DIR / "minimal-example-dataset"
checkpoint_path = MLVC_DIR / "models" / "mlvc-psnr-v1.ckpt"
dataset_dir.mkdir(parents=True, exist_ok=True)

print(f"Using demo dataset from {dataset_dir}")

## Download sample video and model checkpoint

In [ ]:
model_params = {
    "type": "DMC-6.1sb",
    "activation": "LeakyReLU",
    "input_offset": -0.5,
    "feature_channels": 128,
    "spatial_prior_channels": 256,
    "memory_activation": "identity",
    "zero_init_residual": True,
    "chunk_mode": "gated",
    "ffn_gate_activation": "ReLU1",
    "chain_feature_adaptors": True,
}

asset_base_url = "https://mlvideopub.blob.core.windows.net/mlvc/datasets/minimal-example-dataset"
checkpoint_url = "https://mlvideopub.blob.core.windows.net/mlvc/models/mlvc-psnr-v1.ckpt"
video_relative_dir = Path("test-set/yuv/640x360_30fps/s1")
sample_video_filename = "0380a333cb0fef69001fb4260e2a705f_640x360_30fps.yuv"
validation_clips = [
    sample_video_filename,
    "1e88301e03cf356b45d2287c6283de17_640x360_30fps.yuv",
    "3d747f8336ca66bbbb444639d99dd3ce_640x360_30fps.yuv",
    "4d5d7cd504e346a619d38b19b3ec221b_640x360_30fps.yuv",
    "4d8ffedc816b405fe0187bd03392664f_640x360_30fps.yuv",
    "6aa9cf0f933312468ec70baa0fad27f5_640x360_30fps.yuv",
    "6c9c96d33030b3b3879c691760492b14_640x360_30fps.yuv",
]
anchor_relative_path = Path("job-outputs/benchmark_test/anchor/VCD_640x360_30fps_full/intel_hw_hevc_lp.json")


def download_file(url, destination):
    if destination.is_file():
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        dir=destination.parent, prefix=f".{destination.name}.", suffix=".download", delete=False
    ) as temporary_file:
        temporary_path = Path(temporary_file.name)

    try:
        print(f"Downloading {url} to {destination}")
        torch.hub.download_url_to_file(url, temporary_path, progress=True)
        temporary_path.replace(destination)
    finally:
        temporary_path.unlink(missing_ok=True)

    return destination


def download_asset(relative_path):
    relative_path = Path(relative_path)
    url = f"{asset_base_url}/{relative_path.as_posix()}"
    return download_file(url, dataset_dir / relative_path)


checkpoint_path = download_file(checkpoint_url, checkpoint_path)
validation_clips_base_path = dataset_dir / video_relative_dir
test_video_path = download_asset(video_relative_dir / sample_video_filename)

## Helper functions

In [ ]:
def read_yuv420_frames(video_path, width, height):
    video_reader = YUVReader(str(video_path), width, height)
    res = []
    while True:
        y, uv = video_reader.read_one_frame()
        if y is None:
            break
        res.append((y, uv))
    return res


def load_model(model_params, checkpoint_path):
    state_dict = get_state_dict(checkpoint_path)
    model = create_video_model(config=model_params)
    model.load_state_dict(state_dict, strict=True)
    model = model.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    return model


def yuv420_to_rgb(yuv420):
    y = torch.from_numpy(yuv420[0]).float().unsqueeze(0)
    u, v = torch.from_numpy(yuv420[1]).float().unsqueeze(0).chunk(2, dim=1)
    yuv444 = yuv_420_to_444((y, u, v), mode="nearest")
    rgb = ycbcr2rgb(yuv444[0]).cpu().numpy().transpose(1, 2, 0)
    img_pil = Image.fromarray((np.round(rgb * 255.0)).astype(np.uint8))
    return img_pil


def prepare_output(x_hat, padding):
    # Remove padding and convert YUV444 to YUV420
    x_hat_cropped = torch.nn.functional.pad(x_hat, tuple(-x for x in padding))
    y_rec, u_rec, v_rec = yuv_444_to_420(x_hat_cropped.to(torch.float32))
    uv_rec = torch.cat((u_rec, v_rec), dim=1)
    yuv420_rec = (y_rec.cpu().numpy()[0], uv_rec.cpu().numpy()[0])
    return yuv420_rec


def calc_metrics(yuv420_inp, yuv420_rec, encoded_bit_stream):
    def _calc_psnr(x1, x2):
        mse = np.mean((x1 - x2) ** 2)
        if not np.isfinite(mse):
            return -999.9
        if mse < 1e-10:
            return 999.9
        return -10 * np.log10(mse).item()

    y_inp, (u_inp, v_inp) = yuv420_inp
    y_rec, (u_rec, v_rec) = yuv420_rec

    height, width = y_inp.shape[-2:]
    psnr_y = _calc_psnr(y_inp, y_rec)
    psnr_u = _calc_psnr(u_inp, u_rec)
    psnr_v = _calc_psnr(v_inp, v_rec)
    psnr = (6 * psnr_y + psnr_u + psnr_v) / 8
    bpp = (8 * len(encoded_bit_stream)) / (height * width)

    return {
        "psnr_y": psnr_y,
        "psnr_u": psnr_u,
        "psnr_v": psnr_v,
        "psnr": psnr,
        "bpp": bpp,
    }

## ML video codec

In [ ]:
class MlVideoCodec:
    def __init__(self, model):
        self._model = model
        self._dpb = None  # Decoded picture buffer
        self._frame_num = 0
        self._reset_period = 64

    @property
    def fa_idx(self):
        frame_index_map = self._model.frame_index_map
        fa_idx = frame_index_map[self._frame_num % len(frame_index_map)]
        return fa_idx

    @torch.inference_mode()
    def encode(self, yuv420, q_index=42):
        device = next(self._model.parameters()).device
        x, _, padding = prepare_frame(yuv420, is_yuv420=True, precision="fp32", device=device)

        # Generate reference frame
        if self._dpb is None:
            self._dpb = dict(
                ref_frame=torch.ones_like(x) * 0.5,
                ref_feature=None,
            )

        if x.shape != self._dpb["ref_frame"].shape:
            raise ValueError("Input shape does not match the expected shape")

        # Feature reset
        if self._reset_period is not None and self._frame_num % self._reset_period == 1:
            self._dpb["ref_feature"] = None

        result = self._model.compress(x, dpb=self._dpb, q_index=q_index, fa_idx=self.fa_idx, calc_bits_estimates=False)

        # Update the dpb
        self._dpb = result["dpb"]
        self._frame_num += 1

        # Return encoded bit stream and reconstructed frame (YUV420)
        encoded_bit_stream = result["bit_stream"]
        yuv420_rec = prepare_output(result["x_hat"], padding)

        return encoded_bit_stream, yuv420_rec

## Test inference

In [ ]:
# Load model
model = load_model(model_params, checkpoint_path)

# Load test video
test_frames = read_yuv420_frames(test_video_path, 640, 360)

# Encode video
codec = MlVideoCodec(model)
codec_results = []
for i, yuv420_inp in enumerate(tqdm(test_frames, desc="Encoding frames")):
    encoded_bit_stream, yuv420_rec = codec.encode(yuv420_inp)

    codec_results.append(
        {
            "frame_index": i,
            "metrics": calc_metrics(yuv420_inp, yuv420_rec, encoded_bit_stream),
            "img_inp": yuv420_to_rgb(yuv420_inp),
            "img_rec": yuv420_to_rgb(yuv420_rec),
        }
    )

## Plot metrics

In [ ]:
psnrs = np.array([r["metrics"]["psnr"] for r in codec_results])
bpp = np.array([r["metrics"]["bpp"] for r in codec_results])
print(f"Average PSNR: {psnrs.mean():.2f} dB, Average BPP: {bpp.mean():.4f}")

fig, axs = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axs[0].plot(psnrs)
axs[1].plot(bpp)
for ax in axs:
    ax.grid()
axs[0].set_ylabel("PSNR (dB)")
axs[1].set_xlabel("Frame index")
axs[1].set_ylabel("Bits per pixel (BPP)")

## Plot images

In [ ]:
frame_index = 299
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].imshow(codec_results[frame_index]["img_inp"])
axs[0].set_title("Original frame")
axs[1].imshow(codec_results[frame_index]["img_rec"])
axs[1].set_title("Reconstructed frame")
for ax in axs:
    ax.axis("off")

## Optional validation benchmark

In [ ]:
# Download the remaining validation clips and the anchor
for video_filename in validation_clips:
    if video_filename != sample_video_filename:
        download_asset(video_relative_dir / video_filename)
anchor_path = download_asset(anchor_relative_path)

In [ ]:
def run_validation_test(base_path, validation_clips, q_index_list=[0, 21, 42, 63]):
    df_metrics = []
    for video_path, q_index in tqdm(
        itertools.product(validation_clips, q_index_list),
        desc="Processing videos",
        total=len(validation_clips) * len(q_index_list),
    ):
        frames = read_yuv420_frames(base_path / video_path, 640, 360)
        codec = MlVideoCodec(model)
        for i, yuv420_inp in enumerate(frames):
            encoded_bit_stream, yuv420_rec = codec.encode(yuv420_inp, q_index=q_index)
            df_metrics.append(
                {
                    "video_path": video_path,
                    "frame_index": i,
                    "q_index": q_index,
                    **calc_metrics(yuv420_inp, yuv420_rec, encoded_bit_stream),
                }
            )
    df_metrics = pd.DataFrame(df_metrics)
    return df_metrics


def load_anchor_data(anchor_path, filter_clips=None):
    with open(anchor_path) as fin:
        data = json.load(fin)

    df_anchor = []
    for test_class, series_data in data.items():
        for filename, file_data in series_data.items():
            if filter_clips is not None and filename not in filter_clips:
                continue
            for q_index, q_data in file_data.items():
                df_anchor.append(
                    {
                        "test_class": test_class,
                        "video_path": filename,
                        "q_index": int(q_index),
                        "psnr": q_data["ave_all_frame_psnr"],
                        "bpp": q_data["ave_all_frame_bpp"],
                    }
                )
    return pd.DataFrame(df_anchor)


# Run validation test
df_metrics = run_validation_test(validation_clips_base_path, validation_clips)

# Load anchor (H265) data
df_anchor = load_anchor_data(anchor_path, filter_clips=validation_clips)

# Aggregate metrics and calculate BD-rate
df_agg_metrics = df_metrics.groupby(["q_index"]).agg({"psnr": "mean", "bpp": "mean"})
df_agg_anchor = df_anchor.groupby(["q_index"]).agg({"psnr": "mean", "bpp": "mean"})
bd_rate = calculate_bd_rate(df_agg_anchor["bpp"], df_agg_anchor["psnr"], df_agg_metrics["bpp"], df_agg_metrics["psnr"])

# Plot results
plt.figure(figsize=(8, 5))
plt.plot(
    df_agg_metrics["bpp"],
    df_agg_metrics["psnr"],
    marker="o",
    label="ML video codec (BD-rate: %.1f)" % (bd_rate),
)
plt.plot(df_agg_anchor["bpp"], df_agg_anchor["psnr"], marker="o", label="H265")
plt.xlabel("Bits per pixel (BPP)")
plt.ylabel("PSNR (dB)")
plt.title("Rate-distortion curve")
plt.legend()
plt.grid()